In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import sparse
import math  
import sklearn.metrics 
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial import distance_matrix
from scipy.spatial.distance import euclidean, pdist, squareform
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler


In [2]:
def normalizar_datos(matriz_usuario_pelicula):
    # Creamos una copia de la matriz para evitar modificar el original
    matriz_escasez_copy = matriz_usuario_pelicula.copy()
    
    # Inicializamos StandardScaler sin centrado en 0 debido a NaNs
    scaler = StandardScaler(with_mean=True, with_std=True)
    
    # Aplicamos la normalización solo en las columnas que tienen datos no NaN
    for user_id in matriz_escasez_copy.index:
        # Seleccionamos las calificaciones del usuario (excluyendo NaNs)
        user_ratings = matriz_escasez_copy.loc[user_id].dropna()
        if not user_ratings.empty:
            # Normalizamos las calificaciones de este usuario
            normalized_ratings = scaler.fit_transform(user_ratings.values.reshape(-1, 1)).flatten()
            # Colocamos los valores normalizados en la matriz original, manteniendo NaNs donde no hay calificaciones
            matriz_escasez_copy.loc[user_id, user_ratings.index] = normalized_ratings
    
    # Llenamos los NaNs con 0
    matriz_escasez_copy = matriz_escasez_copy.fillna(0)
    return matriz_escasez_copy

In [3]:
def peliculas_del_usuario(usuario, matriz_usuario_pelicula):
    # Filtramos las películas que no tienen valor NaN para el usuario
    usuario_data = matriz_usuario_pelicula.loc[usuario].dropna()

    # Obtenemos las películas y sus valoraciones
    lista_peliculas = usuario_data.index.tolist()  # Películas votadas
    valoraciones_originales = usuario_data.values.tolist()  # Valoraciones correspondientes

    return lista_peliculas, valoraciones_originales


In [4]:
def reescalar_prediccion(prediccion_normalizada, id_usuario, matriz_usuario_pelicula, min_rating=0.5, max_rating=5.0):
    # Calculamos la media y desviación estándar del usuario específico
    media_usuario = matriz_usuario_pelicula.loc[id_usuario].mean(skipna=True)
    desviacion_usuario = matriz_usuario_pelicula.loc[id_usuario].std(skipna=True)
    
    # Si la desviación estándar es válida, aplicamos el reescalado
    if not pd.isna(media_usuario) and not pd.isna(desviacion_usuario):
        prediccion_reescalada = (prediccion_normalizada * desviacion_usuario) + media_usuario
        # Clipping para mantener la predicción dentro del rango y redondeo
        prediccion_reescalada = np.clip(round(prediccion_reescalada, 2), min_rating, max_rating)
    else:
        prediccion_reescalada = np.nan
    
    return prediccion_reescalada

In [5]:
def similitud_distancias(lista_peliculas_usuario, usuario, distancia_escogida, vecinos, minkowski=False, reescalar=False):
    lista_similitud = []
    usuario_objetivo = matriz_normalizada.iloc[[usuario]]  # Datos del usuario objetivo

    for pelicula in lista_peliculas_usuario:
        # Seleccionamos las valoraciones de los usuarios que han visto la película
        usuarios_que_vieron_la_pelicula_con_nulls = matriz_normalizada[pelicula]
        usuarios_que_vieron_la_pelicula = matriz_normalizada[
            usuarios_que_vieron_la_pelicula_con_nulls.notna() & 
            (usuarios_que_vieron_la_pelicula_con_nulls != 0)
        ]
        usuarios_que_vieron_la_pelicula_con_nulls = usuarios_que_vieron_la_pelicula_con_nulls.loc[usuarios_que_vieron_la_pelicula.index]

        # Excluir al usuario objetivo del conjunto de entrenamiento
        usuarios_que_vieron_la_pelicula = usuarios_que_vieron_la_pelicula.drop(index=usuario+1, errors='ignore')
        usuarios_que_vieron_la_pelicula_con_nulls = usuarios_que_vieron_la_pelicula_con_nulls.drop(index=usuario+1, errors='ignore')

        # Ajustar el modelo KNN
        knn = KNeighborsRegressor(metric=distancia_escogida, p=3 if minkowski else None, n_neighbors=vecinos)
        knn.fit(usuarios_que_vieron_la_pelicula, usuarios_que_vieron_la_pelicula_con_nulls)

        # Predecir para el usuario objetivo
        prediccion_usuario = knn.predict(usuario_objetivo)
        
        # Reescalar la predicción
        if reescalar:
            prediccion_usuario = reescalar_prediccion(prediccion_usuario[0], usuario, matriz_usuario_pelicula)
        
        lista_similitud.append(prediccion_usuario)  # Añadir la predicción reescalada

    return lista_similitud

In [6]:
def peliculas_con_minimo_vecinos(lista_peliculas_usuario, valoraciones_peliculas, matriz_normalizada, vecinos_min):
    # Filtramos solo las películas con al menos 'vecinos_min' valoraciones distintas de 0
    peliculas_filtradas = []
    valoraciones_filtradas = []
    
    for i, pelicula in enumerate(lista_peliculas_usuario):
        # Asegurarse de que 'pelicula' es el identificador correcto que corresponde a las columnas de usuario_pelicula
        if pelicula not in matriz_normalizada.columns:
            print(f"Película {pelicula} no encontrada en las columnas.")
            continue
        
        # Seleccionamos los usuarios que vieron la película, es decir, aquellos con valoraciones != 0
        usuarios_que_vieron_la_pelicula = matriz_normalizada[matriz_normalizada[pelicula].notna() & (matriz_normalizada[pelicula] != 0)][pelicula]
        
        # Si el número de usuarios que vieron la película es mayor o igual a 'vecinos_min', la añadimos a la lista
        if len(usuarios_que_vieron_la_pelicula) >= vecinos_min:
            peliculas_filtradas.append(pelicula)
            valoraciones_filtradas.append(valoraciones_peliculas[i])  # Añadimos la valoración correspondiente

    return peliculas_filtradas, valoraciones_filtradas

In [7]:
def calcular_mape(valoraciones_peliculas, similitudes):
    """
    Calcula el MAPE promedio para un conjunto de valoraciones y predicciones.
    """
    # Calcular los errores relativos para cada valoración
    errores_relativos = [
        abs(valoracion - prediccion) / valoracion if valoracion != 0 else 0
        for valoracion, prediccion in zip(valoraciones_peliculas, similitudes)
    ]
    
    # Calcular el MAPE promedio
    mape_promedio = sum(errores_relativos) / len(errores_relativos) * 100 if errores_relativos else 0
    
    return mape_promedio

In [27]:
def total_metrics_randomized(matriz_usuario_pelicula, matriz_normalizada, porcentaje=0.1):
    """
    Calcula el MAPE y RMSE promedio para un conjunto de usuarios seleccionados aleatoriamente en un porcentaje dado.
    """
    minimo_vecinos = 20
    mejores_vecinos = 19

    # Generar una máscara de 10% de posiciones aleatorias donde hay valoraciones originales
    mascara_original = ~matriz_usuario_pelicula.isna()
    num_datos = mascara_original.sum().sum()
    num_datos_a_simular = int(porcentaje * num_datos)

    indices_aleatorios = np.random.choice(
        mascara_original.stack()[mascara_original.stack()].index,
        size=num_datos_a_simular,
        replace=False
    )

    # Crear máscara para las posiciones seleccionadas
    mascara_aleatoria = pd.DataFrame(False, index=matriz_usuario_pelicula.index, columns=matriz_usuario_pelicula.columns)
    for fila, columna in indices_aleatorios:
        mascara_aleatoria.loc[fila, columna] = True

    # Generar una lista aleatoria de usuarios evaluados
    usuarios_evaluados = np.random.permutation(
        mascara_aleatoria.any(axis=1).index[mascara_aleatoria.any(axis=1)]
    )

    usuarios_vistos = 0
    sum_mape = 0  # Suma acumulativa de los MAPE
    sum_rmse = 0  # Suma acumulativa de los RMSE
    total_usuarios = len(usuarios_evaluados)

    for usuario in usuarios_evaluados:
        if usuario not in matriz_usuario_pelicula.index:
            print(f"Usuario {usuario} no encontrado. Saltando.")
            continue

        usuarios_vistos += 1
        print(f"Evaluando usuario: {usuario} ({usuarios_vistos}/{total_usuarios} usuarios evaluados)")

        # Obtener las películas y valoraciones del usuario
        lista_peliculas, valoraciones_peliculas = peliculas_del_usuario(usuario, matriz_usuario_pelicula)

        # Filtrar películas con un mínimo de vecinos
        lista_peliculas, valoraciones_peliculas = peliculas_con_minimo_vecinos(
            lista_peliculas_usuario=lista_peliculas, 
            valoraciones_peliculas=valoraciones_peliculas, 
            matriz_normalizada=matriz_normalizada, 
            vecinos_min=minimo_vecinos
        )

        # Obtener las similitudes calculadas con cosine
        similitud_cosine = similitud_distancias(
            lista_peliculas_usuario=lista_peliculas, 
            usuario=usuario, 
            distancia_escogida='cosine', 
            vecinos=mejores_vecinos,
            reescalar=True
        )

        # Calcular el MAPE promedio para este usuario
        mape_promedio = calcular_mape(valoraciones_peliculas, similitud_cosine)
        sum_mape += mape_promedio  # Sumar el MAPE promedio al total

        # Calcular el RMSE para este usuario
        rmse = mean_squared_error(valoraciones_peliculas, similitud_cosine, squared=False)
        sum_rmse += rmse  # Sumar el RMSE al total

        print(f"Usuario: {usuario}, MAPE: {mape_promedio:.2f}%, RMSE: {rmse:.4f}")


    mape_global = sum_mape / usuarios_vistos
    rmse_global = sum_rmse / usuarios_vistos


    print(f"\nMAPE promedio global: {mape_global:.2f}%" if mape_global is not None else "No se pudo calcular MAPE global.")
    print(f"RMSE promedio global: {rmse_global:.4f}" if rmse_global is not None else "No se pudo calcular RMSE global.")

    return mape_global, rmse_global


In [28]:
df = pd.read_csv("BBDD_100K/ratings.csv")
indice=list(df['userId'].unique())
columnas=list(df['movieId'].unique())
indice=sorted(indice)
columnas=sorted(columnas)
matriz_usuario_pelicula=pd.pivot_table(data=df,values='rating',index='userId',columns='movieId')
matriz_normalizada = normalizar_datos(matriz_usuario_pelicula)


In [29]:
mape_global, rmse_global = total_metrics_randomized(
    matriz_usuario_pelicula=matriz_usuario_pelicula,
    matriz_normalizada=matriz_normalizada
)

Evaluando usuario: 407 (1/593 usuarios evaluados)
Usuario: 407, MAPE: 22.19%, RMSE: 0.9334
Evaluando usuario: 500 (2/593 usuarios evaluados)
Usuario: 500, MAPE: 48.55%, RMSE: 1.1639
Evaluando usuario: 51 (3/593 usuarios evaluados)
Usuario: 51, MAPE: 53.66%, RMSE: 1.1451
Evaluando usuario: 497 (4/593 usuarios evaluados)
Usuario: 497, MAPE: 38.08%, RMSE: 1.2551
Evaluando usuario: 314 (5/593 usuarios evaluados)
Usuario: 314, MAPE: 27.71%, RMSE: 0.8885
Evaluando usuario: 143 (6/593 usuarios evaluados)
Usuario: 143, MAPE: 70.79%, RMSE: 1.5594
Evaluando usuario: 313 (7/593 usuarios evaluados)
Usuario: 313, MAPE: 48.41%, RMSE: 1.3015
Evaluando usuario: 555 (8/593 usuarios evaluados)
Usuario: 555, MAPE: 24.98%, RMSE: 0.9604
Evaluando usuario: 324 (9/593 usuarios evaluados)
Usuario: 324, MAPE: 30.77%, RMSE: 0.8606
Evaluando usuario: 362 (10/593 usuarios evaluados)
Usuario: 362, MAPE: 10.79%, RMSE: 0.5222
Evaluando usuario: 487 (11/593 usuarios evaluados)
Usuario: 487, MAPE: 18.27%, RMSE: 0.6583

KeyboardInterrupt: 

In [41]:
def evaluar_knn_por_semillas(
    matriz_usuario_pelicula, 
    matriz_normalizada, 
    num_ejecuciones=3, 
    porcentaje=0.1
):
    """
    Evalúa el modelo KNN generando diferentes máscaras de simulación usando semillas aleatorias
    y calcula el MAPE y RMSE promedio para cada ejecución.

    Args:
        matriz_usuario_pelicula: DataFrame original de valoraciones de usuarios.
        matriz_normalizada: DataFrame con los datos normalizados.
        num_ejecuciones: Número de ejecuciones con diferentes máscaras aleatorias.
        porcentaje: Porcentaje de datos a simular.

    Returns:
        resultados_df: DataFrame con los resultados de cada ejecución.
        mape_promedio: MAPE promedio de las ejecuciones.
        rmse_promedio: RMSE promedio de las ejecuciones.
    """
    resultados = []

    for seed in range(num_ejecuciones):
        print(f"\n=== Ejecución {seed + 1} ===")
        np.random.seed(seed)
        
        # Calcular métricas aleatorias
        print(f"Generando máscara de simulación con semilla {seed}...")
        mape_global, rmse_global = total_metrics_randomized(
            matriz_usuario_pelicula=matriz_usuario_pelicula,
            matriz_normalizada=matriz_normalizada,
            porcentaje=porcentaje
        )
        print(f"Resultados ejecución {seed + 1} - MAPE: {mape_global:.2f}%, RMSE: {rmse_global:.4f}")
        
        # Guardar resultados por ejecución
        resultados.append({"Ejecución": seed + 1, "MAPE (%)": round(mape_global, 2), "RMSE": round(rmse_global, 4)})

    # Crear DataFrame con los resultados
    resultados_df = pd.DataFrame(resultados)

    # Calcular promedios
    mape_promedio = resultados_df["MAPE (%)"].mean()
    rmse_promedio = resultados_df["RMSE"].mean()

    print("\n=== Resumen Final ===")
    print(f"MAPE promedio: {mape_promedio:.2f}%")
    print(f"RMSE promedio: {rmse_promedio:.4f}")

    return resultados_df, mape_promedio, rmse_promedio


In [ ]:
resultados_df, mape_promedio, rmse_promedio = evaluar_knn_por_semillas(
    matriz_usuario_pelicula=matriz_usuario_pelicula, 
    matriz_normalizada=matriz_normalizada, 
    num_ejecuciones=3, 
    porcentaje=0.1
)


=== Ejecución 1 ===
Generando máscara de simulación con semilla 0...
Evaluando usuario:386
Evaluando usuario:545
Evaluando usuario:122


KeyboardInterrupt: 

In [ ]:
print("\nResultados por ejecución:")
print(resultados_df.to_string(index=False))